# Macro: CPI-U trend and momentum

## What
US CPI-U (`CUSR0000SA0`) from the public BLS data route, with an EWMA(12) trend and a 12-month rate of change.

## Why this model
EWMA shows direction with less noise than the raw index. A 12-month percent change is the usual inflation-style momentum for a monthly price index. `/api/v1/data/macro/series` is not a live route (404).

## How to rerun
Needs only `FINUTIES_API_KEY` in `notebooks/.env`. Run top to bottom.

**Endpoint (verified 200):** `GET /api/v1/data/economic/bls?series_id=CUSR0000SA0`

**Columns used:** `series_id`, `period`, `value`

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


In [2]:
ENDPOINT = "/api/v1/data/economic/bls"
payload = finuties_get(ENDPOINT, {"series_id": "CUSR0000SA0", "limit": 200})
df = pd.DataFrame(normalize_rows(payload))
require_frame(df, ["series_id", "period", "value"], min_rows=13)

df = df[df["series_id"] == "CUSR0000SA0"].copy()
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["date"] = pd.to_datetime(df["period"], errors="coerce")
df = df.dropna(subset=["date", "value"]).sort_values("date").drop_duplicates("date").reset_index(drop=True)
require_frame(df, ["date", "value"], min_rows=13)
require_finite(df["value"], "value")

df["ewma_12"] = df["value"].ewm(span=12, adjust=False).mean()
df["roc_12"] = df["value"].pct_change(12) * 100
require_finite(df["ewma_12"], "ewma_12")
require_finite(df["roc_12"].dropna(), "roc_12")
assert np.isfinite(float(df["value"].iloc[-1]))
assert np.isfinite(float(df["roc_12"].dropna().iloc[-1]))

print(f"CUSR0000SA0 CPI-U  n={len(df)}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df[["date", "value", "ewma_12", "roc_12"]].tail(10)

CUSR0000SA0 CPI-U  n=199
Date range: 2009-11-01 → 2026-06-01


,date,value,ewma_12,roc_12
189,2025-08-01,323.291,319.215837,2.938592
190,2025-09-01,324.245,319.989554,3.022572
191,2025-11-01,325.063,320.770084,2.988300
192,2025-12-01,326.031,321.579456,3.002262
193,2026-01-01,326.588,322.350001,2.828680
194,2026-02-01,327.460,323.136155,2.664589
195,2026-03-01,330.293,324.237208,3.320206
196,2026-04-01,332.407,325.494099,3.947027
197,2026-05-01,333.979,326.799468,4.270033
198,2026-06-01,332.568,327.686935,3.726530


## Charts

Left: CPI-U index level and EWMA(12). Right: 12-month percent change. The API does not send a `unit` for this series; the left axis is the BLS index, the right axis is percent.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(df["date"], df["value"], color="#1f77b4", linewidth=1.7, label="CPI-U index")
axes[0].plot(df["date"], df["ewma_12"], color="#ff7f0e", linewidth=1.7, label="EWMA(12)")
axes[0].set_title("CPI-U level and trend")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Index level (BLS CUSR0000SA0)")
axes[0].legend(loc="upper left")

axes[1].plot(df["date"], df["roc_12"], color="#2ca02c", linewidth=1.7, label="12-month change")
axes[1].axhline(0, color="#444444", linewidth=1, linestyle="--", label="Zero")
axes[1].set_title("12-month momentum")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Percent (%)")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()

## Caveats

- The API `unit` field is null for this series. The y-axis is the BLS CPI-U index level, not a percent.
- Seasonal adjustment and revisions belong to BLS, not FinUties.
- EWMA lags turning points. ROC is not core PCE or a real-time nowcast.
- Not investment advice.